
# MADE 架构复现指导文档

> 论文：*Scalable Deep Learning for RF Fingerprinting: The MADE Architecture for Robust Physical-Layer Device Identification*
> DOI: 10.1109/OJCOMS.2026.3667450
> 作者：Nordine Quadar, Abdellah Chehri, Benoit Debaque（加拿大皇家军事学院 / Thales Defense and Security）

---

## 一、论文核心思想总览

### 1.1 问题背景

RF 指纹识别（RF Fingerprinting）利用无线发射机因制造工艺差异产生的硬件缺陷（功率放大器非线性、本振相位噪声、IQ 失衡、DAC 失真等）作为设备的唯一"指纹"，实现物理层设备认证。现有深度学习方法存在三大根本性缺陷：

1. **闭集假设**：分类器要求在训练时预定义所有设备类别，无法识别未知设备；
2. **可扩展性差**：设备数量增加时，softmax 分类器参数量线性增长，决策边界愈发拥挤，性能指数级下降；
3. **信道鲁棒性不足**：受限于训练时的信道条件，时间变化或接收机位置变化都会导致性能骤降。

### 1.2 MADE 的解决方案

MADE（**MA**sked **D**enoising auto**E**ncoder）通过以下三个核心创新应对上述挑战：

| 挑战 | MADE 的解决方案 |
|------|----------------|
| 闭集假设 | 重建误差异常检测机制（开集识别） |
| 可扩展性 | Transformer 自注意力机制，特征提取层参数量近似常数 |
| 信道鲁棒性 | 循环平稳特征提取（捕获硬件特有的高阶统计特性） |

---

## 二、数据处理流程

### 2.1 数据集

论文使用两个数据集：

#### WiSIG 数据集（WiFi）

- **规模**：174 个商用 WiFi 发射机，41 个 USRP SDR 接收机，约 1000 万个数据包，历时约一个月采集；
- **采样率**：20 MSps，中心频率 2.412 GHz（WiFi 信道 1）；
- **预处理版本**：使用官方"Full WiSig"子集（76.9 GB），已包含包检测、CFO 估计和时序同步；
- **实验配置**（论文选择 3 个单接收机配置）：
  - **RX11**：强多径和干扰，最差场景（6 设备）；
  - **RX22**：改善接收位置，中等质量（6 设备）；
  - **RX77**：良好信号条件，最优场景（5~80 设备，用于可扩展性验证）。

#### UAV 遥控器数据集（无人机控制器）

- **设备**：8 款商用无人机控制器（DJI Matrice、Walkera Q205、DJI Inspire 2、DJI Mini 2、NineEAGLES、Spektrum DX4e、FrSky Taranis、WLtoys）；
- **频段**：2.4 GHz 和 5.8 GHz，使用专有协议；
- **结构**：每设备 19 次独立传输实例（Tx1~Tx19），每次实例包含 1000 万个 IQ 样本，使用 USRP B210 接收；
- **划分**：训练集为 Tx1、Tx2（初始会话），测试集为 Tx3~Tx19（时间跨度 2 小时至 7 天，环境变化）；
- **UAV 预处理步骤**（需自行实现）：
  1. 以设备传输频率为中心的带通滤波；
  2. 自动增益控制（AGC）补偿；
  3. 直流偏置（DC offset）去除。

### 2.2 通用数据标准化

所有数据集均经过以下统一预处理，确保公平比较：

**步骤 1：复数域归一化**

$$s_{norm}[n] = \frac{s_{raw}[n] - \mu_s}{\sigma_s}$$

其中 $s_{raw}[n] = I[n] + jQ[n]$，$\mu_s$ 为样本均值，$\sigma_s$ 为标准差。计算范围为**整个数据集级别**（非单样本级别）。

**步骤 2：序列分帧**

- 固定长度序列：$L = 256$ 个样本；
- 划分为 8 个 Patch，每个 Patch 包含 32 个样本；
- IQ 两通道作为双通道输入，保留相位关系。

**步骤 3：数据集划分**

- 分层划分（保持各类别比例）：70% 训练 / 20% 验证 / 10% 测试；
- 使用 3 个不同随机种子进行独立实验，计算方差和置信区间。

### 2.3 循环平稳特征提取

**窗口参数**：

- 窗口长度：4096 个样本；
- 相邻窗口重叠：20%；
- 计算二阶、四阶、六阶统计量，然后下采样对齐 Patch 结构。

**三阶统计量的计算公式**：

二阶特征（捕捉振荡器相位噪声和功率放大器非线性）：

$$I^{(2)} = I^2_{norm}, \quad Q^{(2)} = Q^2_{norm}$$

四阶特征（捕捉不可见于低阶统计的非线性交叉项）：

$$I^{(4)} = (I^{(2)})^2 - (Q^{(2)})^2$$
$$Q^{(4)} = 2I^{(2)}Q^{(2)}$$

六阶特征（捕捉功率放大器失真产生的复杂非线性，抑制高斯噪声）：

$$I^{(6)} = (I^{(2)})^3 - 3I^{(2)}(Q^{(2)})^2$$
$$Q^{(6)} = 3(I^{(2)})^2 Q^{(2)} - (Q^{(2)})^3$$

**关键设计原理**：高斯噪声不具有循环平稳性，随着统计阶次升高，噪声贡献趋近于零，而硬件特征保持显著。六阶时噪底大幅降低。

**双路径处理**：

- **路径 A（RMS 归一化）**：将 IQ 信号变换到单位功率，强调波形形状，消除幅度变化；
- **路径 B（保留原始信号）**：保留绝对幅度信息（可能是独特指纹，归一化后会丢失）。

两条路径的特征经过以下 4 个专用硬件特征提取器：

| 提取器 | 捕捉的硬件特性 |
|--------|---------------|
| 前端缺陷提取器 | IQ 失衡、DC 偏置、增益失配 |
| 振荡器特性提取器 | 相位噪声、频率稳定性、抖动模式 |
| 放大器签名提取器 | 非线性失真、包络变化 |
| 滤波器响应提取器 | 频率响应异常、群延迟变化 |

提取的特征随后通过嵌入层映射到 Transformer 所需的维度空间（$d_{model} = 128$）。

---

## 三、网络模型（MADE 架构）

MADE 由四个相互关联的核心模块组成：特征预处理流水线、掩码去噪自编码器、Transformer 注意力模块、分类头。

### 3.1 架构全貌

```
输入 IQ 样本
    │
    ├──[路径1: IQ模式]──────────────────────────────────────────────────────┐
    │   归一化 → 分帧(L=256) → 分Patch(8×32) → Patch嵌入(→128维)           │
    │                                                                        │
    └──[路径2: Cyclo模式]────────────────────────────────────────────────────┤
        归一化信号 + 原始信号 → 4个硬件特征提取器                           │
        → 特征融合与投影(→128维)                                             │
                                                                             ↓
                                           ┌──────────────────────────────────┐
                                           │         MADE核心模块              │
                                           │  ┌──────────┐  ┌──────────┐     │
                                           │  │ 掩码模块  │  │ 去噪模块  │     │
                                           │  │(pmask=0.25)│  │(σ=0.05)  │     │
                                           │  └──────────┘  └──────────┘     │
                                           │         ↓ 叠加                   │
                                           │   ┌──────────────────┐           │
                                           │   │ MLP编码器         │           │
                                           │   │ 128→256→128       │           │
                                           │   │ BatchNorm+GELU    │           │
                                           │   │ Dropout(p=0.1)    │           │
                                           │   └──────────────────┘           │
                                           │         ↓                         │
                                           │   ┌──────────────────┐           │
                                           │   │ 位置编码(正弦)    │           │
                                           │   └──────────────────┘           │
                                           │         ↓                         │
                                           │   ┌──────────────────┐           │
                                           │   │ Transformer模块   │           │
                                           │   │ Layer1:           │           │
                                           │   │  8头自注意力      │           │
                                           │   │  前馈网络         │           │
                                           │   │  层归一化+残差    │           │
                                           │   │ Layer2:           │           │
                                           │   │  8头自注意力      │           │
                                           │   │  前馈网络         │           │
                                           │   │  层归一化+残差    │           │
                                           │   └──────────────────┘           │
                                           │         ↓                         │
                                           │   ┌──────────────────┐           │
                                           │   │ 注意力池化        │           │
                                           │   │ (→固定128维嵌入)  │           │
                                           │   └──────────────────┘           │
                                           └──────┬──────────────┬──────────────┘
                                                  ↓              ↓
                                      ┌──────────────┐  ┌──────────────┐
                                      │  分类头      │  │  重建头      │
                                      │128→256→128   │  │128→256→输入维│
                                      │→Nclasses     │  │ (MSE损失)    │
                                      │(交叉熵损失)   │  └──────────────┘
                                      └──────────────┘
```

### 3.2 各模块详细说明

#### 模块一：掩码去噪自编码器（MADE 核心）

**掩码操作**（预训练阶段）：

$$\mathbf{x}_{masked} = \mathbf{M} \odot \mathbf{x} + (1 - \mathbf{M}) \odot \mathbf{t}_{mask}$$

- $\mathbf{M}$：二值掩码，掩码概率 $p_{mask} = 0.25$；
- $\mathbf{t}_{mask}$：可学习的掩码 Token（替换被掩码的 Patch）；
- 在 Patch 级别而非样本级别进行掩码（硬件签名跨越多个样本）；
- **重要发现**：消融实验证明掩码会损害性能 4.6%，与 NLP 中 BERT 的成功截然相反，因为 RF 信号缺乏语义冗余，时序连续性至关重要。**微调阶段禁用掩码**。

**去噪操作**（预训练阶段）：

$$\mathbf{x}_{corrupted} = \mathbf{x}_{masked} + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma_{noise}^2 \mathbf{I})$$

- 噪声标准差 $\sigma_{noise} = 0.05$；
- 这是与掩码不同的策略：通过引入受控噪声而非删除信息；**微调阶段禁用去噪**。

**MLP 编码器**：

$$z = f_{enc}(x_{corrupted}) = D(\phi(B(W_2 \cdot \phi(B(W_1 x)))))$$

- $\phi(\cdot)$：GELU 激活函数（梯度比 ReLU 更平滑）；
- $B(\cdot)$：批归一化（稳定训练，隐式正则化）；
- $D(\cdot)$：Dropout，$p = 0.1$；
- 维度变化：$d_{input} \rightarrow 256 \rightarrow 128$（瓶颈层强制压缩）。

**MLP 解码器**（镜像结构）：

$$\hat{x} = f_{dec}(z) = W_4 \cdot \phi(B(W_3 z))$$

- 维度变化：$128 \rightarrow 256 \rightarrow d_{input}$；
- 对称设计保证潜空间充分表达时信息可被恢复。

#### 模块二：Transformer 注意力模块

**多头自注意力**（$h = 8$，$d_k = d_v = 16$，2 层）：

$$A(Z) = [head_1 \| \cdots \| head_h] W^O$$

- $W^O \in \mathbb{R}^{(h \cdot d_v) \times d_{model}}$：输出投影矩阵；
- 每层集成残差连接和层归一化；
- 不同注意力头可能专注于不同特征：如一个头关注跨 Patch 的相位一致性，另一个头关注幅度模式。

**正弦位置编码**（固定式）：

$$PE_{(pos, 2i)} = \sin\left(pos \cdot 10000^{-2i/d_{model}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(pos \cdot 10000^{-2i/d_{model}}\right)$$

- RF 信号的时序排列至关重要（相位连续性依赖采样顺序）；
- 正弦编码可泛化到比训练时更长的序列。

**注意力池化**（替代简单平均池化）：

$$\alpha_i = \frac{\exp(\mathbf{w}^\top \tanh(\mathbf{h}_i))}{\sum_{j=1}^{N_p} \exp(\mathbf{w}^\top \tanh(\mathbf{h}_j))}$$

$$\mathbf{h}_{pool} = \sum_{i=1}^{N_p} \alpha_i \mathbf{h}_i$$

- $\mathbf{h}_i$：第 $i$ 个 Patch 的 Transformer 输出；
- $\mathbf{w}$：可学习的注意力权重向量；
- $\tanh$ 非线性防止单个 Patch 主导（有界注意力分数）；
- 产生固定大小的 128 维设备嵌入，无论输入长度如何变化。

#### 模块三：分类头（三层全连接）

$$\mathbf{h}_1 = D(\sigma(B(W_{c,1} \mathbf{h}_{pool})))$$
$$\mathbf{h}_2 = D(\sigma(B(W_{c,2} \mathbf{h}_1)))$$
$$\mathbf{y}_{pred} = \text{Softmax}(W_{c,3} \mathbf{h}_2)$$

- $\sigma(\cdot)$：ReLU 激活；
- Dropout $p = 0.3$（防止对训练特定样本过拟合，对跨传输泛化至关重要）；
- 维度变化：$128 \rightarrow 256 \rightarrow 128 \rightarrow N_{classes}$；
- 扩展到 256 维提供额外的非线性决策边界学习能力。

#### 参数规模

| 模块 | 参数量 |
|------|--------|
| Transformer 编码器（固定，与设备数无关）| ~1.8M |
| MADE 模块 | ~0.2M |
| 分类头（随 $N_{classes}$ 线性增长）| < 1% 总量 |
| **总计** | **~2.1M** |

---

## 四、训练策略

### 4.1 双阶段训练范式

MADE 采用两阶段训练，核心逻辑是"先学会理解信号，再学会区分设备"：

#### 阶段一：预训练（表示学习）

**目标**：学习信号的鲁棒表示，而非最大化分类准确率。

**超参数**：

| 参数 | 值 |
|------|----|
| 初始化 | $\theta \sim \mathcal{N}(0, 0.02)$（稳定梯度流） |
| 掩码概率 $p_{mask}$ | 0.25 |
| 去噪强度 $\sigma_{noise}$ | 0.05 |
| 损失权重 $\lambda_{recon} / \lambda_{cls}$ | 0.8 / 0.2 |
| 优化器 | AdamW，$lr = 5 \times 10^{-5}$，$\text{weight\_decay} = 0.01$ |
| 最大 Epoch | 50 |
| Early Stopping | 验证重建损失连续 10 个 Epoch 无改善则停止 |
| Batch Size | 32 |

**为什么用 AdamW 而非 Adam**：AdamW 的解耦权重衰减比传统 L2 惩罚提供更强的正则化，对学习可泛化表示而非记忆训练样本至关重要。

**损失函数**：

$$\mathcal{L}_{total} = \lambda_{recon} \cdot \mathcal{L}_{recon} + \lambda_{cls} \cdot \mathcal{L}_{cls}$$

重建损失（MSE）：

$$\mathcal{L}_{recon} = \frac{1}{N} \sum_{i=1}^{N} \|\mathbf{x}_i - \hat{\mathbf{x}}_i\|_2^2$$

分类损失（交叉熵）：

$$\mathcal{L}_{cls} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{k=1}^{K} y_{i,k} \log(\hat{y}_{i,k})$$

预训练时强调重建（$\lambda_{recon}=0.8$），防止网络坍缩为只能分类而无法泛化的平凡解。

#### 阶段二：微调（分类优化）

**目标**：在预训练表示基础上，以分类为主要目标进行精调。

**关键变化**：

- **禁用掩码和去噪**：保证网络处理干净信号，与推理条件一致；
- **反转损失权重**：$\lambda_{recon} = 0.2$，$\lambda_{cls} = 0.8$；
- **差异化学习率**（防止灾难性遗忘）：

| 模块 | 学习率 | 比例 |
|------|--------|------|
| 编码器（保护预训练知识）| $1 \times 10^{-5}$ | 1× |
| Transformer 注意力层 | $5 \times 10^{-5}$ | 5× |
| 分类头（学习新决策边界）| $10 \times 10^{-5}$ | 10× |

- **Epoch 数**：20（预训练的强初始化使得收敛更快）；
- **重建损失作为正则器**：防止编码器坍缩为仅服务于分类的窄判别特征，维持信号域的有意义表示。

### 4.2 开集识别机制

MADE 的重建分支自然支持开集识别，无需额外训练：

- 已知设备：解码器能够精确重建 → 低重建误差；
- 未知设备（训练时未见）：潜空间表示发散，解码器重建失败 → 高重建误差；
- 设置重建误差阈值即可实现异常检测，无需 Siamese 网络对或隐藏状态直方图分析。

当前验证显示 MSE 主要集中在 0.05~0.15 之间，具有紧密聚类特性，为阈值设置提供了基础。

### 4.3 完整训练流程（伪代码）

```
输入：数据集 D = {(x_i, y_i)}，i=1..N
输出：训练参数 θ_final

# 阶段一：预训练
初始化 θ ~ N(0, 0.02)
设置 p_mask = 0.25, σ_noise = 0.05
for epoch in range(50):
    for batch B in D:
        对 B 应用随机 Patch 掩码（概率 0.25）
        对剩余 Patch 添加高斯噪声（σ=0.05）
        前向传播：B_hat, y_pred = f_θ(B_noisy)
        计算损失：L = 0.8 * L_recon + 0.2 * L_cls
        AdamW 反向传播更新 θ（lr=5e-5）
    if 验证重建损失连续10 epoch无改善:
        break（Early Stopping）

# 阶段二：微调
禁用掩码和去噪
设置差异化学习率（编码器1×, Transformer 5×, 分类头 10×）
for epoch in range(20):
    for batch (B, Y) in D:
        前向传播：B_hat, y_pred = f_θ(B_clean)
        计算损失：L = 0.2 * L_recon + 0.8 * L_cls
        差异化学习率反向传播更新
返回 θ_final
```

---

## 五、实验结果分析

### 5.1 消融研究（WiSIG RX11，6 设备）

在最困难的干扰场景下，系统性测试各组件贡献：

| 配置 | P1（循环平稳）| P2（去噪）| P3（掩码）| 准确率 |
|------|:---:|:---:|:---:|--------|
| 基线 | OFF | OFF | OFF | 56.2 ± 0.6% |
| 仅循环平稳 | ON | OFF | OFF | 63.9 ± 0.5% |
| 仅去噪 | OFF | ON | OFF | 61.9 ± 0.5% |
| 仅掩码 | OFF | OFF | ON | 51.6 ± 0.7% |
| 全部特征 | ON | ON | ON | 71.7 ± 0.4% |
| **最优（P1+P2）** | **ON** | **ON** | **OFF** | **71.5 ± 0.4%** |

关键发现：

- **循环平稳特征贡献最大**（+7.7%）：有效捕获跨传输持久存在的硬件特定缺陷；
- **去噪额外贡献 +5.7%**：通过重建学习强调稳定信号特征；
- **掩码损害性能 −4.6%**：RF 信号缺乏 NLP 中的语义冗余，时序连续性不可间断；
- P1+P2 组合（不含掩码）为最优配置，训练时间约 66 分钟（基线 45 分钟，开销 47%）。

### 5.2 多数据集综合评估

| 数据集 | 设备数 | 准确率 | 精确率 | 召回率 | F1 |
|--------|--------|--------|--------|--------|-----|
| WiSIG RX11（强干扰）| 6 | 71.5 ± 0.7% | 69.0% | 71.5% | 65.9% |
| WiSIG RX22（改善位置）| 6 | 83.5 ± 0.4% | 84.1% | 83.5% | 80.8% |
| WiSIG RX77（最优条件）| 6 | 99.9 ± 0.2% | 99.9% | 99.9% | 99.9% |
| UAV 控制器（跨传输）| 8 | 92.9 ± 0.4% | 94.5% | 92.9% | 92.7% |

三个 WiSIG 配置揭示了 40% 的性能变化区间，凸显了信号质量对 RF 指纹识别的关键影响。

UAV 测试（训练 Tx1~Tx2，测试 Tx3~Tx19，时间跨度 2 小时至 7 天）92.9% 的准确率验证了架构在真实操作条件下的时序泛化能力。

### 5.3 可扩展性分析

| 设备数 | 准确率 | 精确率 | 召回率 | F1 | 退化幅度 |
|--------|--------|--------|--------|-----|----------|
| 5 | 80.2 ± 0.5% | 71.7% | 80.2% | 73.9% | 基准 |
| 20 | 72.3 ± 0.6% | 71.5% | 72.3% | 69.9% | −7.9% |
| 40 | 76.5 ± 0.4% | 75.3% | 76.5% | 75.3% | −3.7% |
| 80 | 75.2 ± 0.5% | 74.3% | 75.2% | 74.4% | −5.0% |

**非单调规律**（重要发现）：

- 5→20 设备：最大下降（−7.9%），类别混淆增加；
- 20→40 设备：性能**恢复 +4.2%**，更大种群提供更多特征空间覆盖，使决策边界更鲁棒；
- 40→80 设备：轻微下降（−1.3%），趋于稳定；
- **16 倍设备规模扩展，总退化仅 5.0%**，远优于传统分类器的指数衰减。

**计算资源扩展性**：

| 指标 | 表现 |
|------|------|
| GPU 显存 | 线性增长，约 0.12 GB/设备（5 设备 6.2 GB → 80 设备 15.8 GB）|
| 训练时间 | 亚线性增长，$O(n^{0.68})$（5 设备 12 分钟 → 80 设备 69 分钟）|
| 推理延迟 | 稳定在 12~15 ms（不随设备数变化）|

### 5.4 重建质量分析

- 基线配置：PSNR = 18.72 dB，最终重建损失 = 0.0538 MSE；
- 训练收敛快速：损失从约 0.6 降至 0.0538，仅需约 20 个 Epoch；
- IQ 两分量性能均衡：MSE(I) = 0.0536，MSE(Q) = 0.0540，保证幅度和相位信息均被保留；
- 重建误差集中在 0.05~0.15 之间，紧密聚类，为未知设备检测阈值设置提供依据。

### 5.5 基线对比

| 架构 | 层配置 | 参数量 | RX77 准确率 | UAV 准确率 | 80 设备准确率 |
|------|--------|--------|-------------|------------|--------------|
| 传统 CNN | Conv(64,128,256) + FC(256,128) | 1.2M | 91.2% | 78.3% | 61.2% |
| LSTM | LSTM(128 隐层, 2层) + FC(256,128) | 0.8M | 89.6% | 76.9% | 58.4% |
| CNN-LSTM 混合 | Conv(64,128) + LSTM(128,1层) + FC(256,128) | 1.5M | 94.3% | 85.9% | 67.3% |
| **MADE（本文）** | **见第三章** | **2.1M** | **99.9%** | **92.9%** | **75.2%** |
| 改善幅度 | | | +5.6% | +7.0% | +7.9% |

所有基线均训练 50 Epoch，$lr = 10^{-4}$，相同数据划分和评估协议。

---

## 六、讨论与关键洞察

### 6.1 为什么掩码对 RF 信号失效

NLP 中的掩码学习成功依赖**语义冗余**：文字的上下文可以约束被掩码的 Token。RF 信号的硬件指纹完全不同：

- **载波频率偏移（CFO）**：在整个数据包上表现为持续相位漂移，需要连续观察；
- **放大器非线性**：在特定幅度阈值处的特征失真，需要完整幅度包络；
- **振荡器抖动**：引入跨越数百个样本的相关扰动，需要时序连续性。

掩码打断了这些连续时序模式，反而使模型无法区分被随机掩码的正常信号和真实的硬件特征。**去噪则相反**：保持时序完整性，只添加可学习区分的随机噪声。

### 6.2 循环平稳特征的优势

从物理原理理解循环平稳特征的效果：

- 通信信号的统计特性随时间周期性变化（由于载波调制、符号时序、帧结构）；
- 硬件缺陷以设备特定的方式调制这些周期性模式，嵌入唯一签名到循环域；
- 高阶循环统计量（四阶、六阶）在高斯噪声下的贡献趋近于零，而硬件特征保持显著；
- 因此循环平稳特征在 SNR 波动环境中比原始 IQ 处理更鲁棒。

### 6.3 局限性与未来工作方向

| 局限性 | 说明 | 未来方向 |
|--------|------|----------|
| 跨接收机泛化 | 实验使用单接收机配置，未测试跨接收机场景 | 系统性跨接收机评估 |
| 协议多样性 | 仅验证 WiFi 和 UAV 控制器 | 扩展至 LoRa、蓝牙、5G/6G |
| 对抗攻击鲁棒性 | 未评估对抗样本场景 | 数字和物理对抗攻击测试 |
| 大规模扩展性 | 仅测试至 80 设备，~150 设备内存即满 | 层次化聚类、知识蒸馏、联邦学习 |
| 边缘部署 | 当前需要 8.4 GB 显存，12~15 ms 延迟 | 模型压缩、量化、FPGA/ASIC 实现 |
| 开集验证 | 重建异常检测假设未在真实未知设备上验证 | 使用真正分布外设备进行验证 |

---

## 七、复现实施要点总结

### 7.1 环境配置

- GPU：NVIDIA RTX 3090（24 GB 显存）或同等配置；
- 框架：PyTorch（推荐）；
- 最小批大小：32（平衡 GPU 利用率和梯度多样性）。

### 7.2 关键超参数汇总

| 参数 | 预训练 | 微调 |
|------|--------|------|
| 学习率 | $5 \times 10^{-5}$ | $1\sim10 \times 10^{-5}$（差异化）|
| Epoch | 50 | 20 |
| 掩码比例 | 0.25 | 0.0（禁用）|
| 去噪强度 | 0.05 | 0.0（禁用）|
| $\lambda_{recon}$ | 0.8 | 0.2 |
| $\lambda_{cls}$ | 0.2 | 0.8 |
| Weight Decay | 0.01 | 0.01 |
| Batch Size | 32 | 32 |
| Early Stopping | 10 Epoch | — |

### 7.3 模型关键维度

| 参数 | 值 |
|------|----|
| 序列长度 $L$ | 256 |
| Patch 数 $N_p$ | 8 |
| Patch 大小 | 32 |
| 嵌入维度 $d_{model}$ | 128 |
| Transformer 层数 | 2 |
| 注意力头数 $h$ | 8 |
| $d_k = d_v$ | 16 |
| 编码器瓶颈维度 | 128 |
| 分类头中间维度 | 256 |
| Dropout（编码器）| 0.1 |
| Dropout（分类头）| 0.3 |

### 7.4 复现难点与注意事项

1. **双路径预处理**：IQ 模式和循环平稳模式需要独立实现，特征融合投影层需保证输出统一为 128 维；
2. **循环平稳特征计算**：需正确实现二阶、四阶、六阶统计量的计算，特别注意六阶计算中的符号；
3. **掩码 Token 可学习性**：$\mathbf{t}_{mask}$ 是可训练参数而非固定值；
4. **差异化学习率**：微调阶段编码器、Transformer、分类头使用不同学习率，需通过优化器的参数组机制实现（PyTorch 中的 `param_groups`）；
5. **损失权重切换**：两阶段损失权重不同，需在代码中显式切换；
6. **重建目标**：解码器的目标是原始干净信号（不含掩码和噪声），而不是被破坏的输入；
7. **数据集准备**：WiSIG 的 "Full WiSig" 预处理子集（76.9 GB），已含 CFO 估计和时序同步，可直接使用；UAV 数据集的带通滤波等预处理需自行实现；
8. **早停条件**：基于**验证重建损失**而非验证分类准确率触发早停；
9. **循环平稳窗口**：4096 样本窗口，20% 重叠，需下采样对齐至 8 个 Patch 结构；
10. **可扩展性评估**：使用嵌套子集 $D_5 \subset D_{20} \subset D_{40} \subset D_{80}$，而非独立随机采样。

---

## 八、结论

MADE 通过三大创新（循环平稳特征提取、去噪自编码器预训练、双目标训练范式）克服了传统 RF 指纹识别方法的核心限制。消融实验清晰表明，掩码在 RF 域不适用，而循环平稳特征 + 去噪的组合是最优方案。论文实现了在 16 倍设备规模扩展下仅 5.0% 的受控性能退化，以及 12~15 ms 的稳定推理延迟，为下一代无线安全应用建立了可扩展基础。
